# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook cho **Kaggle** (khuyên dùng: *Save & Run All* chạy hết trong một session 12 giờ) hoặc Colab. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Clone fork vào thư mục làm việc (/kaggle/working trên Kaggle, /content trên Colab). Đổi REPO_DIR nếu đặt chỗ khác.
REPO_DIR = "K4-D02-NguyenVanQuocViet-2A202602973"
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/vietvuivui/{REPO_DIR}.git
CODE_DIR = f"{REPO_DIR}/submissions/2A202602973_NguyenVanQuocViet/code"
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, CODE_DIR)                 # code/ của bạn (đã chép từ starter/)

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
LABELS_DIR = "data/labels"
# Tự tìm thư mục chứa ảnh .jpg sau khi giải nén ("data/images" hoặc "data"), kiểm tra bằng một tên file trong CSV.
_probe = pd.read_csv(f"{LABELS_DIR}/train_subset0.csv").Filename.iloc[0]
IMAGES_DIR = next(d for d in ["data/images", "data"] + [f"data/{x}" for x in sorted(os.listdir("data"))]
                  if os.path.exists(f"{d}/{_probe}"))
print("IMAGES_DIR =", IMAGES_DIR)

ON_KAGGLE = os.path.isdir("/kaggle/working")
ON_COLAB = os.path.isdir("/content") and not ON_KAGGLE
# Kaggle: mọi thứ trong /kaggle/working được giữ lại ở Output sau "Save & Run All".
# Colab: hay bị ngắt -> lưu lên Drive. Chạy lại notebook: thí nghiệm đã xong được nạp lại, không train lại.
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
RUNS_DIR = ("/kaggle/working/deepweeds_runs" if ON_KAGGLE else
            "/content/drive/MyDrive/deepweeds_runs" if ON_COLAB else "runs")
CACHE_DIR = "/tmp/deepweeds_cache" if ON_KAGGLE else "/content/cache" if ON_COLAB else "cache"  # đĩa cục bộ, không lưu
SUB_DIR = os.path.dirname(CODE_DIR)          # submissions/2A202602973_NguyenVanQuocViet
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tham số dùng chung cho mọi Config. Dự đoán val của B/T nằm trong RUNS_DIR; chỉ chung kết + mốc (Bước 4)
# ghi vào SUB_DIR/predictions để nộp. cache_dir: ảnh giải mã một lần thành memmap (đọc nhanh hơn JPEG).
PATHS = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=f"{RUNS_DIR}/runs",
             pred_dir=f"{RUNS_DIR}/predictions", curves_dir=f"{RUNS_DIR}/curves", cache_dir=CACHE_DIR,
             num_workers=min(4, os.cpu_count() or 2))

# Ghi đè công thức nền cho MỌI thí nghiệm (B, T, F). 12 epoch = mặc định GUIDE (10-15), vừa một session Kaggle 12 giờ.
TRAIN_OVERRIDES = dict(epochs=12)
print("RUNS_DIR =", RUNS_DIR, "| device =", device, "| num_workers =", PATHS["num_workers"], "|", TRAIN_OVERRIDES)

def sync_outputs():
    """Chép ảnh biểu đồ từ RUNS_DIR sang SUB_DIR/curves (để commit nộp bài)."""
    import shutil, glob
    os.makedirs(f"{SUB_DIR}/curves", exist_ok=True)
    for f in glob.glob(f"{RUNS_DIR}/curves/*.png"):
        shutil.copy2(f, f"{SUB_DIR}/curves/")
    print("Đã chép", len(glob.glob(f"{SUB_DIR}/curves/*.png")), "ảnh vào", f"{SUB_DIR}/curves")

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# 0.1 — Đọc và kiểm tra split (README mục 2.1). check_split dừng bằng assert nếu vi phạm.
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image
import dataset as D

train_df, val_df, test_df = D.load_split(LABELS_DIR, fold=0)
labels_df = D.load_labels(LABELS_DIR)                     # labels.csv gốc: đối chiếu nhãn (S1)
split_stats = D.check_split(train_df, val_df, test_df, IMAGES_DIR, labels_df=labels_df)
print("\nKẾT LUẬN: giao rỗng:", all(v == 0 for v in split_stats["overlap"].values()),
      "| hợp =", split_stats["union"], "| thiếu file:", split_stats["missing"],
      "| nhãn lệch labels.csv:", split_stats["label_mismatch"], "| cảnh báo tỉ lệ:", split_stats["warnings"] or "không")

In [ ]:
# 0.2 — EDA: phân bố lớp, đối chiếu Table 1 bài báo, tỉ lệ mất cân bằng, kích thước/kênh ảnh
PAPER = {"Chinee Apple": 1125, "Lantana": 1064, "Parkinsonia": 1031, "Parthenium": 1022, "Prickly Acacia": 1062,
         "Rubber Vine": 1009, "Siam Weed": 1074, "Snake Weed": 1016, "Negatives": 9106}
per_class = pd.DataFrame(split_stats["per_class"])
per_class["paper"] = pd.Series(PAPER)
per_class["khớp bài báo"] = per_class["total"] == per_class["paper"]
per_class["% tổng"] = (100 * per_class["total"] / per_class["total"].sum()).round(2)
display(per_class)
imb = per_class["total"].max() / per_class["total"].min()
print(f"Tỉ lệ lớp nhiều nhất / ít nhất = {per_class['total'].idxmax()} {per_class['total'].max()} / "
      f"{per_class['total'].idxmin()} {per_class['total'].min()} = {imb:.2f}")

ax = per_class[["train", "val", "test"]].plot.bar(stacked=True, figsize=(10, 4),
                                                  title="Phân bố lớp DeepWeeds theo tập (fold 0)")
ax.set_ylabel("số ảnh"); plt.xticks(rotation=30, ha="right"); plt.tight_layout()
os.makedirs(f"{SUB_DIR}/figures", exist_ok=True)
plt.savefig(f"{SUB_DIR}/figures/eda_class_distribution.png", dpi=130); plt.show()

# Kích thước và số kênh (chỉ đọc header của mọi ảnh)
all_files = pd.concat([train_df, val_df, test_df]).Filename
shapes = Counter()
for f in all_files:
    with Image.open(f"{IMAGES_DIR}/{f}") as im:
        shapes[(im.size, im.mode)] += 1
print("Kích thước (W, H) và mode:", dict(shapes))

# Thống kê màu trên 500 ảnh train (tham khảo; vẫn chuẩn hoá theo mean/std của trọng số tiền huấn luyện)
arr = np.stack([np.asarray(Image.open(f"{IMAGES_DIR}/{f}").convert("RGB"), dtype=np.float32) / 255
                for f in train_df.Filename.sample(500, random_state=0)])
print("mean RGB train ~", arr.mean((0, 1, 2)).round(3), "| std ~", arr.std((0, 1, 2)).round(3),
      "| ImageNet:", D.IMAGENET_MEAN, D.IMAGENET_STD)

In [ ]:
# 0.3 — Xem 4 ảnh mỗi lớp: cặp loài nào dễ nhầm, Negative trông ra sao
fig, axes = plt.subplots(D.NUM_CLASSES, 4, figsize=(8, 19))
for c in range(D.NUM_CLASSES):
    for j, f in enumerate(train_df[train_df.Label == c].sample(4, random_state=0).Filename):
        axes[c, j].imshow(Image.open(f"{IMAGES_DIR}/{f}")); axes[c, j].set_xticks([]); axes[c, j].set_yticks([])
    axes[c, 0].set_ylabel(D.CLASS_NAMES[c], fontsize=9)
plt.tight_layout(); plt.savefig(f"{SUB_DIR}/figures/eda_samples.png", dpi=110); plt.show()
# TODO (báo cáo): ghi nhận xét — ví dụ Chinee Apple vs Snake Weed, Parkinsonia vs Prickly Acacia.

In [ ]:
# 0.4 — Kiểm tra pipeline (GUIDE mục 1.3) trên resnet50
import math, torch.nn.functional as F
import model as M, losses as L
from train import set_seed, evaluate

set_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = M.build_model("resnet50", num_classes=9, init="finetune").to(device)
cfgm = m.pretrained_cfg
print("trọng số:", m.weights_tag, "| mean/std:", cfgm["mean"], cfgm["std"])
tf_tr = D.build_transforms(True, 224, "basic", cfgm["mean"], cfgm["std"])
tf_ev = D.build_transforms(False, 224, mean=cfgm["mean"], std=cfgm["std"])

# (1) seed: hai loader cùng seed cho cùng thứ tự batch và cùng augmentation (kể cả trong worker)
def first_batch(seed):
    set_seed(seed)
    ld = D.make_loader(train_df, IMAGES_DIR, tf_tr, batch_size=16, train=True, num_workers=2, seed=seed)
    return next(iter(ld))
a, b = first_batch(0), first_batch(0)
assert a[2] == b[2] and torch.equal(a[0], b[0]), "seed không tái lập được batch đầu"
print("(1) seed tái lập thứ tự + augmentation: OK")
PIPELINE_CHECKS = {"seed_reproducible": True}

# (2) loss ban đầu của head mới ~ -ln(1/9) = 2.197 (trung bình trên 256 ảnh val)
val_ld = D.make_loader(val_df.sample(256, random_state=0), IMAGES_DIR, tf_ev, batch_size=64, train=False)
_, _, _, loss0 = evaluate(m, val_ld, torch.nn.CrossEntropyLoss(), device, amp=False)
print(f"(2) loss ban đầu {loss0:.3f} (kỳ vọng ~{-math.log(1/9):.3f})")
PIPELINE_CHECKS["initial_loss"] = round(loss0, 4)

# (3) overfit một batch 32 ảnh (transform eval, không augmentation) tới loss gần 0
x, y, _ = next(iter(D.make_loader(train_df, IMAGES_DIR, tf_ev, batch_size=32, train=True, num_workers=0)))
x, y = x.to(device), y.to(device)
M.set_train_mode(m)
opt = torch.optim.AdamW(M.param_groups(m, 1e-4, 1e-3, 0.0))
for step in range(60):
    loss = F.cross_entropy(m(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 10 == 0 or step == 59:
        print(f"(3) step {step:2d} loss {loss.item():.4f}")
PIPELINE_CHECKS.update(overfit_batch=32, overfit_steps=60, overfit_final_loss=round(loss.item(), 5))
assert loss.item() < 0.1, "không overfit được 1 batch -> kiểm tra code/model trước khi chạy tiếp"

# (4) ảnh sau augmentation (đã giải chuẩn hoá) + nhãn; hàng dưới: CutMix trộn cả ảnh lẫn nhãn
ds_aug = D.DeepWeedsDataset(train_df.sample(8, random_state=1), IMAGES_DIR, tf_tr)
mean, std = torch.tensor(cfgm["mean"])[:, None, None], torch.tensor(cfgm["std"])[:, None, None]
show = lambda t: (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
xs = torch.stack([ds_aug[i][0] for i in range(8)]); ys = torch.tensor([ds_aug[i][1] for i in range(8)])
xm, (ya, yb, lam) = L.mix_batch(xs, ys, 1.0, "cutmix")
fig, axes = plt.subplots(2, 8, figsize=(16, 4.8))
for i in range(8):
    axes[0, i].imshow(show(xs[i])); axes[0, i].set_title(D.CLASS_NAMES[ys[i]], fontsize=8); axes[0, i].axis("off")
    axes[1, i].imshow(show(xm[i])); axes[1, i].axis("off")
    axes[1, i].set_title(f"{D.CLASS_NAMES[ya[i]][:9]} /\n{D.CLASS_NAMES[yb[i]][:9]}", fontsize=7)
fig.suptitle(f"Trên: augmentation 'basic' + nhãn. Dưới: CutMix (nhãn a / b), lam = {lam:.2f}")
plt.tight_layout(); plt.show()

# (5) train()/eval(): evaluate() phải tắt BN/Dropout; frozen phải giữ BN ở eval khi train
m.train(); evaluate(m, val_ld, None, device, amp=False)
assert not any(mod.training for mod in m.modules()), "evaluate() không gọi model.eval()"
_, _, l1, _ = evaluate(m, val_ld, None, device, amp=False)
_, _, l2, _ = evaluate(m, val_ld, None, device, amp=False)
assert np.allclose(l1, l2, atol=1e-4), "hai lần eval khác nhau -> còn ngẫu nhiên (Dropout/BN) lúc eval"
mf = M.build_model("resnet50", num_classes=9, init="frozen")
M.set_train_mode(mf)
bns = [mod for mod in mf.modules() if isinstance(mod, torch.nn.BatchNorm2d)]
assert mf.get_classifier().training and not any(bn.training for bn in bns)
print(f"(5) evaluate() -> eval: OK | frozen: head ở train, {len(bns)} BN ở eval: OK")
PIPELINE_CHECKS.update(augmentation_plot="ô 0.4, hình (4)", eval_mode_ok=True, frozen_bn_eval=len(bns))
print("PIPELINE_CHECKS =", PIPELINE_CHECKS)
del m, mf, opt; torch.cuda.empty_cache()

In [ ]:
# 0.5 — Bộ test của repo (eval.py + khung) và test tự viết của code/ (focal γ=0, CutMix, param_groups, BN, ...)
!cd {REPO_DIR} && python -m unittest discover -s tests
!cd {CODE_DIR} && python -m unittest -v test_code 2>&1 | tail -30

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed 0, mỗi backbone một lần chạy, mã `B01`–`B05`:
ResNet-50 (mốc), ConvNeXt-T, DeiT-S (transformer), EfficientNet-B0 và MobileNetV3-L (nhẹ).

**Tag trọng số được ghim** (`X.BACKBONES`), tất cả chỉ tiền huấn luyện trên ImageNet-1k để so sánh công bằng: `resnet50.a1_in1k`, `convnext_tiny.fb_in1k`, `deit_small_patch16_224.fb_in1k`, `efficientnet_b0.ra_in1k`, `mobilenetv3_large_100.ra_in1k` (mặc định của `convnext_tiny` là `in12k_ft_in1k` – nhiều dữ liệu hơn). Val/test: resize 256 rồi CenterCrop 224; chuẩn hoá theo mean/std của từng bộ trọng số.

**Quy tắc chọn (chỉ dùng val):** các backbone có macro-F1 val cách mức cao nhất không quá `NOISE_F1 = 0.005`
coi là *không phân biệt được* (mới 1 seed); trong nhóm đó chọn backbone có độ trễ batch 1 thấp nhất.
Muốn chọn khác, đặt `BACKBONE_OVERRIDE` và ghi lý do vào báo cáo.

**Ước lượng thời gian:** sau epoch đầu của `B01`, xem cột `epoch_time_s` (in ở mỗi epoch). Tổng ≈ 20 lần train × 12 epoch × thời gian/epoch (backbone nhẹ nhanh hơn ResNet/DeiT).

In [ ]:
import experiments as X, timm
print("Ngưỡng quy tắc chọn: NOISE_F1 =", X.NOISE_F1, "| INFER_MIN_GAIN =", X.INFER_MIN_GAIN)
for _, bb in X.BACKBONES:                          # bằng chứng tag có trong timm phiên bản đang dùng
    arch = bb.split(".")[0]
    print(f"{bb:36s} có sẵn: {timm.list_pretrained(arch + '.*')}")

bb_df = X.step1_backbones(PATHS, device, **TRAIN_OVERRIDES)   # đã xong thì bỏ qua (nạp lại)
bb_df, bb_corr = X.backbone_analysis(bb_df, PATHS)            # hội tụ, quá khớp, tương quan FLOPs-độ trễ
display(bb_df.sort_values("val_macro_f1", ascending=False))
print("Tương quan thứ hạng (5 điểm, tham khảo):", {k: round(v, 2) for k, v in bb_corr.items()})
X.plot_backbones(bb_df, f"{SUB_DIR}/figures/backbones.png")
X.plot_backbone_curves(bb_df, PATHS, f"{SUB_DIR}/figures/backbones_curves.png")

BACKBONE_OVERRIDE = None
BACKBONE, why_bb = X.select_backbone(bb_df, BACKBONE_OVERRIDE)
print("Backbone đi tiếp:", BACKBONE, "\n", why_bb)
sync_outputs()

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Trên backbone đã chọn. Nguyên tắc 1: mỗi `T0x` khác `T00` **đúng một yếu tố** (cùng backbone, cùng 12 epoch,
cùng seed 0, cùng split). `T00` seed 0 chính là lần chạy ở Bước 1 (được dùng lại).

**Nguyên tắc 2 – tự đo nhiễu:** chạy `T00` thêm seed 1 và 2 (đây cũng là mốc `T00` của Bước 4, nên tổng số lần
train không tăng). Ngưỡng nhiễu = max(std macro-F1 val qua 3 seed, 0,003). Mỗi dòng được ghi `vs_noise`:
*vượt nhiễu*, *không phân biệt được* hoặc *kém hơn rõ*.

| Trục | Thí nghiệm (khác T00 ở) | Câu hỏi |
|---|---|---|
| A. Khởi tạo | T01 đóng băng backbone, chỉ train head | Đóng băng có đủ không? |
| B. Augmentation | T02 TrivialAugment · T03 lật dọc + xoay 90° · T04 CutMix | Phép nào giúp, phép nào hại? Lật dọc có hợp lệ? |
| C. Loss | T05 label smoothing 0,1 · T06 focal γ=2 · T07 CE trọng số 1/n (từ train) | Loss nào cải thiện F1 lớp hiếm? |
| D. Cân bằng mẫu | T09 sampler cân bằng lớp | Sampler khác loss có trọng số thế nào (so T07)? |
| F. Chính quy hoá | T08 EMA 0,995 (đánh giá bằng trọng số EMA) | EMA có giúp "miễn phí" không? |

**Kết hợp `T10`:** mọi `T0x` so với **cùng** `T00` (không đổi nền giữa chừng, nên thứ tự trục không ảnh hưởng).
Mỗi trục lấy biến thể có Δ lớn nhất; ghép các trục thắng rõ (Δ > ngưỡng nhiễu). Ít hơn 2 trục như vậy thì nới
dần (Δ > 0, rồi 2 trục Δ lớn nhất) để vẫn kiểm tra được hiệu ứng cộng dồn hay triệt tiêu.
**Công thức chung kết:** dòng seed 0 có macro-F1 val cao nhất trong `T00`–`T10`.

In [ ]:
NOISE = X.measure_noise(PATHS, BACKBONE, **TRAIN_OVERRIDES)        # T00 seed 0 (dùng lại), 1, 2
print(f"Nhiễu T00: macro-F1 val {[round(v, 4) for v in NOISE['val_macro_f1']]} -> std {NOISE['std']:.4f}, "
      f"ngưỡng {NOISE['threshold']:.4f}")

tr_df = X.step2_training(PATHS, BACKBONE, noise=NOISE, **TRAIN_OVERRIDES)   # 9 ablation, seed 0
COMBO, why_combo = X.select_combination(tr_df)
print(why_combo, COMBO)
combo_s = X.step2_combo(PATHS, BACKBONE, COMBO, **TRAIN_OVERRIDES)
tr_df = X.add_combo_row(tr_df, combo_s, why_combo, noise=NOISE)
display(tr_df)

RECIPE_ID, RECIPE, why_recipe = X.select_recipe(tr_df, combo=COMBO)
print("Công thức chung kết:", RECIPE_ID, RECIPE, "\n", why_recipe)
sync_outputs()

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Trên **val**, với model của công thức đã chọn (seed 0), **không huấn luyện lại**:

| Mã | Phương pháp | Chi phí suy luận |
|---|---|---|
| `I00` | 1 view (resize 256 + center crop 224), mốc | 1 lượt |
| `I01` / `I03a` | TTA lật ngang, gộp xác suất / gộp logit | K = 2 |
| `I02` / `I03b` | TTA 5 crop + lật, gộp xác suất / gộp logit | K = 10 |
| `I04_256/288/320` | độ phân giải kiểm tra (FixRes) | 1 lượt, FLOPs tăng |
| `I05` | ensemble 3 lần chạy tốt nhất Bước 1–2 (khác backbone/công thức) | 3 mô hình |
| `I05_seeds` | ensemble 3 seed của T00 | 3 mô hình |
| `I06_soup` | model soup: trung bình trọng số 3 seed của T00 | 1 lượt |
| `I06` / `I06_raw` | trọng số EMA vs trọng số thường, **cùng lần chạy T08, cùng epoch** | 1 lượt |
| `I07` | temperature scaling (T khớp trên val), ECE trước/sau | 1 lượt |
| `I08_*` | gộp BN (ghi rõ nếu kiến trúc không có BN), AMP, FP16 | 1 lượt |

Độ trễ (`benchmark.py`): warmup 10, `cuda.synchronize` trước và sau, 100 lần, p50/p95/p99 ở batch 1, thông lượng ở
batch 32, chỉ đo forward (không tính tiền xử lý). Cột `cost_class` và `realtime_ok` (p95 ≤ 100 ms) dùng để trả lời
nhận định của slide: TTA/ensemble hợp ngoại tuyến, robot nên dùng thứ không tốn thêm.

**Quy tắc chọn cho chung kết:** trong các phương pháp *một mô hình* (I00–I04), lấy macro-F1 val cao nhất nếu hơn
`I00` ít nhất `INFER_MIN_GAIN = 0.002`, ngược lại giữ `I00`. Ensemble không chọn vì chung kết phải train lại
≥ 3 seed cho mọi thành viên (vượt ngân sách GPU). Temperature scaling luôn áp dụng thêm.

In [ ]:
S = lambda exp_id, seed=0: X.run_summary(PATHS, exp_id, seed)
candidates = X.ensemble_candidates(PATHS, bb_df, tr_df, k=3)
inf_df, lat_df = X.step3_inference(S(RECIPE_ID), candidates, device, f"{RUNS_DIR}/step3_inference.csv",
                                   ema_summary=S("T08"), seed_runs=NOISE["runs"])
display(inf_df)
display(lat_df)
X.plot_tradeoff(inf_df, f"{SUB_DIR}/figures/tradeoff.png")
TRADEOFF = X.inference_tradeoff_summary(inf_df)
print(TRADEOFF)

METHOD_OVERRIDE = None
METHOD, why_inf = X.select_inference(inf_df, METHOD_OVERRIDE)
print("Suy luận cho chung kết:", METHOD, "\n", why_inf)

## Bước 4 — Chung kết: 3 seed, test một lần mỗi seed
`F01` = backbone + công thức đã chọn, seed 0/1/2 (seed 0 dùng lại lần chạy ở Bước 2). Mốc `T00` cũng chạy seed 0/1/2.
`X.final_predict` chạy val và **test đúng một lần** cho mỗi seed rồi ghi vào `predictions/` của bài nộp:

| File | Nội dung |
|---|---|
| `F01_seed<k>_val.csv`, `F01_seed<k>_test.csv` | phương pháp đã chọn + temperature scaling (T khớp trên val của seed đó) |
| `F01uncal_seed<k>_test.csv` | như trên nhưng chưa temperature scaling (chấm I4a) |
| `F01rt_seed<k>_test.csv` | cấu hình thời gian thực: cùng model, 1 view (chấm I5) |
| `T00_seed<k>_val.csv`, `T00_seed<k>_test.csv` | mốc: công thức nền + 1 view |

Nếu file test đã có, hàm **không** chạy lại. Sau khi xem kết quả test, không quay lại sửa cấu hình.

In [ ]:
FINAL_PRED = f"{SUB_DIR}/predictions"
final_runs = X.step4_final(PATHS, BACKBONE, RECIPE, METHOD, FINAL_PRED, device, seeds=(0, 1, 2),
                           **TRAIN_OVERRIDES)  # 2 lần train mới: F01 seed 1, 2 (F01 seed 0 và T00 mọi seed đã có)
fin_df = X.final_table(FINAL_PRED, configs=X.final_configs(BACKBONE, RECIPE_ID, RECIPE, METHOD))
display(fin_df)
RT_P95, RT_SRC = X.realtime_p95(inf_df)
print(f"Cấu hình thời gian thực F01rt: p95 batch 1 = {RT_P95:.1f} ms ({RT_SRC})")
sync_outputs()

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (không sửa eval.py), cho MỌI nhóm file trong predictions/
EVAL_OUT = f"{RUNS_DIR}/eval_out"
for tag in ["F01", "T00", "F01rt", "F01uncal"]:          # test: đối chiếu test_subset0.csv
    !python {REPO_DIR}/eval.py score --pred "{FINAL_PRED}/{tag}_seed*_test.csv" --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag {tag} --out {EVAL_OUT}
for tag in ["F01", "T00"]:                                # val: đối chiếu val_subset0.csv
    !python {REPO_DIR}/eval.py score --pred "{FINAL_PRED}/{tag}_seed*_val.csv" --test-csv {LABELS_DIR}/val_subset0.csv --labels {LABELS_DIR}/labels.csv --tag {tag}_val --out {EVAL_OUT}

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận)
grade_out = !python {REPO_DIR}/eval.py grade --final "{FINAL_PRED}/F01_seed*_test.csv" --baseline "{FINAL_PRED}/T00_seed*_test.csv" --uncal "{FINAL_PRED}/F01uncal_seed*_test.csv" --final-val "{FINAL_PRED}/F01_seed*_val.csv" --val-csv {LABELS_DIR}/val_subset0.csv --latency-p95-ms {RT_P95} --latency-method proper --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out {EVAL_OUT}
grade_text = "\n".join(grade_out)
print(grade_text)

## Bước 5 — Sản phẩm
Ghi vào thư mục bài nộp: `results.xlsx` (7 sheet theo GUIDE mục 6.1), `figures/` (backbone, đánh đổi suy luận,
ma trận nhầm lẫn, ảnh bị đoán sai), `curves/`, `predictions/`, và **bản nháp** `report.md` có sẵn bảng số liệu
thật; phần nhận xét (TODO) bạn tự viết. Sau đó `git add` / `commit` / `push` thư mục `submissions/`.

In [ ]:
FIG = f"{SUB_DIR}/figures"
cm_f01 = X.plot_confusion(FINAL_PRED, "F01", f"{FIG}/confusion_F01.png")
X.plot_confusion(FINAL_PRED, "T00", f"{FIG}/confusion_T00.png")
CONFUSIONS = X.top_confusions(cm_f01, k=5)                 # các cặp bị nhầm nhiều nhất (tổng các seed)
display(CONFUSIONS)
X.plot_errors(f"{FINAL_PRED}/F01_seed0_test.csv", IMAGES_DIR, f"{FIG}/errors_chinee_snake.png")
top = CONFUSIONS.iloc[0]
X.plot_errors(f"{FINAL_PRED}/F01_seed0_test.csv", IMAGES_DIR, f"{FIG}/errors_top_pair.png",
              pairs=((int(top.true_idx), int(top.pred_idx)), (int(top.pred_idx), int(top.true_idx))))
pc_df = X.per_class_table(EVAL_OUT)
display(X.hard_class_table(pc_df))                         # precision/recall/F1 Chinee Apple, Snake Weed

sheets = {"Backbones": bb_df, "Training": tr_df, "Inference": inf_df, "Final": fin_df, "PerClass": pc_df,
          "Latency": lat_df, "Summary": X.summary_table(bb_df, tr_df, inf_df, fin_df)}
X.write_results_xlsx(f"{SUB_DIR}/results.xlsx", sheets)    # tô nổi bật dòng tốt nhất mọi sheet (DEFAULT_HIGHLIGHT)
sync_outputs()

# Kiểm tra trước khi nộp: số khớp eval.py, mọi lần train có ảnh curves
PROBLEMS = X.consistency_check(fin_df, pc_df, EVAL_OUT)
print("Khớp số Final/PerClass với eval.py:", "OK" if not PROBLEMS else PROBLEMS)
MISSING = X.check_curves(PATHS["out_dir"], f"{SUB_DIR}/curves")
print("Ảnh curves thiếu:", MISSING or "không")

CONTRIB = X.contributions(bb_df, tr_df, inf_df, BACKBONE, RECIPE_ID, METHOD, fin_df, NOISE)
display(CONTRIB)
EXP_INDEX = X.experiment_index(PATHS["out_dir"])
ENV = X.env_info(); print(ENV)
X.write_report_draft(f"{SUB_DIR}/report.md", env=ENV, split_stats=globals().get("split_stats"),
                     pipeline_checks=globals().get("PIPELINE_CHECKS"), bb_corr=bb_corr, noise=NOISE, backbone=BACKBONE,
                     bb_df=bb_df, why_bb=why_bb, tr_df=tr_df, why_combo=why_combo, recipe_id=RECIPE_ID,
                     why_recipe=why_recipe, inf_df=inf_df, method=METHOD, why_inf=why_inf, fin_df=fin_df, pc_df=pc_df,
                     rt_p95=RT_P95, rt_src=RT_SRC, epochs=TRAIN_OVERRIDES.get("epochs", 12),
                     final_desc=X.final_configs(BACKBONE, RECIPE_ID, RECIPE, METHOD)["F01"],
                     tradeoff=TRADEOFF, confusions=CONFUSIONS, contrib=CONTRIB, exp_index=EXP_INDEX,
                     consistency=PROBLEMS, grade_text=grade_text)
!rm -f data/images.zip   # bớt ~490 MB khỏi Output của Kaggle (ảnh đã giải nén vẫn còn)
!ls -la {SUB_DIR} {SUB_DIR}/predictions {SUB_DIR}/figures